# Benchmark Analysis — Prompting vs Schema RAG vs QLoRA on Spider

Reads a report written by `python -m src.evaluation.compare_approaches` — by default the run named in
`reports/LATEST`; set the `REPORT_DIR` environment variable to analyse another run.

1. Headline metrics (table)
2. Execution accuracy by mode
3. Execution accuracy by Spider hardness — *where* does fine-tuning help?
4. Latency (P50 → P95)
5. Where failures come from
6. Which questions flip between modes — raw material for the case study's failure analysis

Requires `pip install pandas matplotlib` on top of `requirements.txt`. Charts follow one rule throughout:
**a mode always has the same colour**, so colour never needs re-learning between charts.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORTS = ROOT / "reports"
if os.environ.get("REPORT_DIR"):
    REPORT_DIR = Path(os.environ["REPORT_DIR"])
else:
    REPORT_DIR = REPORTS / (REPORTS / "LATEST").read_text().strip()

summary = json.loads((REPORT_DIR / "summary.json").read_text())
meta, results = summary["meta"], summary["results"]
ALL_MODES = ["prompt", "rag", "qlora", "qlora_rag"]
MODES = [m for m in ALL_MODES if m in results]
LABELS = {
    "prompt": "Base + Prompting",
    "rag": "Base + Schema RAG",
    "qlora": "QLoRA",
    "qlora_rag": "QLoRA + Schema RAG",
}
preds = pd.concat(
    [pd.read_json(REPORT_DIR / f"predictions_{m}.jsonl", lines=True) for m in MODES],
    ignore_index=True,
)
print(
    f"Report {REPORT_DIR.name}: split={meta['split']}  n={meta['n']}  base={meta['base_model']}  "
    f"adapter={meta['adapter']}  repairs={meta['max_repairs']}  modes={MODES}"
)

In [ ]:
# Validated palette (dataviz reference instance; both themes pass the CVD / contrast checks).
THEME = "light"  # "dark" uses its own steps, validated against the dark surface
PALETTE = {
    "light": dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781",
                  grid="#e1e0d9", axis="#c3c2b7",
                  series=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]),
    "dark": dict(surface="#1a1a19", ink="#ffffff", ink2="#c3c2b7", muted="#898781",
                 grid="#2c2c2a", axis="#383835",
                 series=["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181"]),
}[THEME]
# Colour follows the entity, never its rank: each mode owns a fixed slot.
MODE_COLOR = dict(zip(ALL_MODES, PALETTE["series"]))
GAP = 1.5  # ~2 px surface-coloured edge = the spacer between adjacent fills

plt.rcParams.update({
    "figure.facecolor": PALETTE["surface"], "axes.facecolor": PALETTE["surface"],
    "savefig.facecolor": PALETTE["surface"], "text.color": PALETTE["ink"],
    "axes.labelcolor": PALETTE["ink2"], "xtick.color": PALETTE["muted"],
    "ytick.color": PALETTE["ink2"], "axes.edgecolor": PALETTE["axis"],
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "grid.color": PALETTE["grid"], "grid.linewidth": 0.6,
    "grid.linestyle": "-", "axes.axisbelow": True, "font.family": "sans-serif",
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "legend.frameon": False, "figure.dpi": 110,
})


def percent_axis(ax, axis="x"):
    fmt = plt.FuncFormatter(lambda v, _: f"{v:.0%}")
    (ax.xaxis if axis == "x" else ax.yaxis).set_major_formatter(fmt)

## 1. Headline metrics

In [ ]:
rows = []
for m in MODES:
    s = results[m]
    rows.append({
        "Approach": LABELS[m],
        "Exec Acc": s["execution_accuracy"], "Exact Match": s["exact_match"],
        "Valid SQL": s["valid_sql_rate"], "Hallucination": s["hallucination_rate"],
        "Unsafe generated": s["unsafe_generated_rate"],
        "P50 ms": s["latency_ms_p50"], "P95 ms": s["latency_ms_p95"],
        "Prompt tok": s["prompt_tokens_mean"], "$ / 1k q": s["cost_per_1k_queries_usd"],
    })
headline = pd.DataFrame(rows).set_index("Approach")
headline.style.format({
    "Exec Acc": "{:.1%}", "Exact Match": "{:.1%}", "Valid SQL": "{:.1%}",
    "Hallucination": "{:.1%}", "Unsafe generated": "{:.1%}",
    "P50 ms": "{:,.0f}", "P95 ms": "{:,.0f}", "Prompt tok": "{:,.0f}", "$ / 1k q": "${:.3f}",
})

## 2. Execution accuracy by mode

Execution accuracy is the primary metric: predicted and gold SQL return the same result.
With n≈1,000, differences under ~3 points are within noise (95% CI ≈ ±2.8 pts at 70%).

In [ ]:
ex = [results[m]["execution_accuracy"] for m in MODES]
fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(MODES) + 1.0))
y = list(range(len(MODES)))[::-1]
ax.barh(y, ex, height=0.35, color=[MODE_COLOR[m] for m in MODES],
        edgecolor=PALETTE["surface"], linewidth=GAP)
for yi, v in zip(y, ex):
    ax.text(v + 0.01, yi, f"{v:.1%}", va="center", color=PALETTE["ink"], fontsize=10)
ax.set_yticks(y, [LABELS[m] for m in MODES])
ax.set_xlim(0, 1.0)
percent_axis(ax)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
ax.set_title(f"Execution accuracy - Spider {meta['split']} (n={meta['n']})")
fig.tight_layout()
plt.show()

## 3. Accuracy by Spider hardness

Fine-tuning is expected to pay off most on *hard* / *extra* queries (nested queries, set operations,
multi-join aggregation); retrieval mostly helps schema linking and literals. The table below the chart
carries the exact values and bucket sizes.

In [ ]:
levels = ["easy", "medium", "hard", "extra"]
by_h = pd.DataFrame(
    {LABELS[m]: [results[m]["by_hardness"].get(h, {}).get("execution_accuracy") for h in levels]
     for m in MODES},
    index=[h.capitalize() for h in levels],
)
counts = {h: results[MODES[0]]["by_hardness"].get(h, {}).get("n", 0) for h in levels}

fig, ax = plt.subplots(figsize=(8, 3.6))
width = min(0.8 / len(MODES), 0.12)  # cap columns at ~24 px
for i, m in enumerate(MODES):
    xs = [j + (i - (len(MODES) - 1) / 2) * width for j in range(len(levels))]
    ax.bar(xs, by_h[LABELS[m]].fillna(0), width=width, color=MODE_COLOR[m], label=LABELS[m],
           edgecolor=PALETTE["surface"], linewidth=GAP)
ax.set_xticks(range(len(levels)), [f"{h.capitalize()}\n(n={counts[h]})" for h in levels])
ax.set_ylim(0, 1.0)
percent_axis(ax, "y")
ax.grid(axis="x", visible=False)
ax.tick_params(axis="x", length=0)
ax.legend(ncols=len(MODES), loc="upper right", bbox_to_anchor=(1, 1.16), fontsize=9)
ax.set_title("Execution accuracy by Spider hardness", pad=24)
fig.tight_layout()
plt.show()
by_h.style.format("{:.1%}", na_rep="-")

## 4. Latency: P50 → P95

Shape = statistic (circle P50, diamond P95), colour = mode. Log scale: generation dominates, and the
non-LLM stages (retrieval, validation, execution) are milliseconds — see the table below.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 0.55 * len(MODES) + 1.3))
y = list(range(len(MODES)))[::-1]
for yi, m in zip(y, MODES):
    p50, p95 = results[m]["latency_ms_p50"], results[m]["latency_ms_p95"]
    ax.plot([p50, p95], [yi, yi], color=MODE_COLOR[m], linewidth=1.5, zorder=2)
    ax.scatter([p50], [yi], s=60, marker="o", color=MODE_COLOR[m], zorder=3,
               edgecolors=PALETTE["surface"], linewidths=GAP)
    ax.scatter([p95], [yi], s=70, marker="D", color=MODE_COLOR[m], zorder=3,
               edgecolors=PALETTE["surface"], linewidths=GAP)
    label = f"P95 {p95 / 1000:.1f} s" if p95 >= 1000 else f"P95 {p95:.0f} ms"
    ax.text(p95 * 1.08, yi, label, va="center", color=PALETTE["ink2"], fontsize=9)
ax.set_xscale("log")
ax.xaxis.set_major_locator(plt.LogLocator(subs=(1, 2, 5)))
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1000:g} s" if v >= 1000 else f"{v:g} ms"))
lo = min(results[m]["latency_ms_p50"] for m in MODES)
hi = max(results[m]["latency_ms_p95"] for m in MODES)
ax.set_xlim(lo / 1.6, hi * 2.6)
ax.set_yticks(y, [LABELS[m] for m in MODES])
ax.set_ylim(-0.6, len(MODES) - 0.4)
ax.tick_params(axis="y", length=0)
ax.grid(axis="y", visible=False)
ax.set_xlabel("End-to-end latency per question (log scale)")
handles = [
    plt.Line2D([], [], marker="o", linestyle="", color=PALETTE["muted"], label="P50"),
    plt.Line2D([], [], marker="D", linestyle="", color=PALETTE["muted"], label="P95"),
]
ax.legend(handles=handles, ncols=2, loc="upper right", bbox_to_anchor=(1, 1.22), fontsize=9)
ax.set_title("Latency spread by mode", pad=22)
fig.tight_layout()
plt.show()

stage = preds.groupby("mode")[["latency_ms", "generation_ms"]].median().reindex(MODES)
stage["non-LLM overhead ms"] = stage["latency_ms"] - stage["generation_ms"]
stage.index = [LABELS[m] for m in stage.index]
stage.round(1)

## 5. Where failures come from

Each panel is one failure type, as a share of all questions. *Wrong result* = valid SQL that ran but
returned a different answer (semantic error); *rejected* = blocked by the safety layer (hallucinated
identifiers, unsafe statements); *exec error* = the database refused it; *no answer* = `CANNOT_ANSWER`
or backend error.

In [ ]:
def outcome(row):
    if row["exec_match"]:
        return "correct"
    return {"ok": "wrong result", "rejected": "rejected", "exec_error": "exec error"}.get(
        row["status"], "no answer"
    )


preds["outcome"] = preds.apply(outcome, axis=1)
kinds = ["wrong result", "rejected", "exec error", "no answer"]
share = (
    preds.groupby("mode")["outcome"].value_counts(normalize=True).unstack(fill_value=0)
    .reindex(index=MODES, columns=["correct", *kinds], fill_value=0)
)

top = max(share[kinds].max().max(), 0.05)
fig, axes = plt.subplots(1, len(kinds), figsize=(11, 0.45 * len(MODES) + 1.3), sharey=True)
y = list(range(len(MODES)))[::-1]
for ax, kind in zip(axes, kinds):
    vals = share[kind].tolist()
    keep = [i for i, v in enumerate(vals) if v > 0]  # no zero-width stubs
    ax.barh([y[i] for i in keep], [vals[i] for i in keep], height=0.35,
            color=[MODE_COLOR[MODES[i]] for i in keep], edgecolor=PALETTE["surface"], linewidth=GAP)
    ax.set_title(kind.capitalize(), fontsize=10)
    ax.set_xlim(0, top * 1.15)
    percent_axis(ax)
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", length=0)
axes[0].set_yticks(y, [LABELS[m] for m in MODES])
fig.suptitle("Failure breakdown (share of all questions)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
plt.show()
share.rename(index=LABELS).style.format("{:.1%}")

## 6. Which questions flip between modes

A contingency table, not a chart: how many questions the best mode fixes vs breaks relative to the
prompting baseline. Fix/break counts tell you more than the net delta.

In [ ]:
best = max(MODES, key=lambda m: results[m]["execution_accuracy"])
base = "prompt" if "prompt" in MODES else MODES[0]
wide = preds.pivot_table(index="idx", columns="mode", values="exec_match", aggfunc="first").astype(bool)
verdict = {True: "correct", False: "wrong"}
flips = pd.crosstab(
    wide[base].map(verdict), wide[best].map(verdict),
    rownames=[LABELS[base]], colnames=[LABELS[best]],
)
print(f"baseline = {LABELS[base]}   best = {LABELS[best]}")
flips

In [ ]:
cols = ["question", "gold_sql", "pred_sql", "hardness"]
by_mode = {m: preds[preds["mode"] == m].set_index("idx") for m in (base, best)}
fixed = wide.index[~wide[base] & wide[best]]
broken = wide.index[wide[base] & ~wide[best]]

pd.set_option("display.max_colwidth", 120)
print(f"Fixed by {LABELS[best]}: {len(fixed)}   Broken by {LABELS[best]}: {len(broken)}")
by_mode[best].loc[fixed[:10], cols]

In [ ]:
# Regressions are the most instructive rows for the case study's failure-mode section
by_mode[best].loc[broken[:10], cols].join(
    by_mode[base].loc[broken[:10], ["pred_sql"]].rename(columns={"pred_sql": f"{base}_sql"})
)

## Using the results

- Copy `summary.md` from the report directory into the README "Benchmark Results" table.
- Paste the hardness table and one fixed/broken example per failure type into `docs/CASE_STUDY.md` §13–14.
- CV bullet: execution accuracy of `prompt` (X) → `qlora_rag` (Y), from the same run.